# TSFS12 Hand-in exercise 5: Data-Driven Motion Prediction
This exercise utilizes data from the SIND dataset to train a deep neural network for motion prediction. The dataset consists of several scenarios with various agents moving about different signalized intersections (see images below). The goal is to predict the future positions of the agents based on their past positions and velocities, taking agent-to-agent interactions into account. Current version of the prediction models uses 3 seconds of historical data and targets predicting up to 5 seconds of future trajectories. Data is recorded at 5 Hz.

This exercise relies heavily on Python and PyTorch libraries. For the basic parts of this exercise, you are not expected to have any prior knowledge of either Python or PyTorch. However, you are encouraged to explore the documentation of these libraries to gain a deeper understanding of the code if you have the required prior knowledge. You will be provided with a set of pre-trained models, and your task is to evaluate and experiment with the models on a test set and analyze and discuss the results.

The data used are real recorded data from 3 densely trafficked intersections in China described in *SIND: A Drone Dataset at Signalized Intersection in China*: https://arxiv.org/abs/2209.02297.

<div style="text-align: center;">
    <img src="media/Chongqing_NR.png" alt="Chongqing_NR" style="width: 30%;">
    <img src="media/Changchun_Pudong.png" alt="Chongqing_NR"style="width: 30%;">
    <img src="media/Xi'an_Shanglin.png" alt="Chongqing_NR" style="width: 30%;">
</div>

The data is preprocessed for you to facilitate this exercise. All preprocessing was done using the *Dronalize* toolbox: https://github.com/westny/dronalize toolbox.

If you are running this notebook from the university linux-labs, the data is directly available (more instructions below). If you are running from your own computer, the preprocessed data can be downloaded from the Lisam course page. The full raw data is available at https://github.com/SOTIF-AVLab/SinD.

If you install on your own computer, some of the packages require specifications specific to your machine.
* https://pytorch.org/get-started/locally/
* https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html
* https://github.com/rusty1s/pytorch_cluster

Follow the instructions for each package

In addition, you will need to install the pure python-packages
* torchinfo
* lightning

For example, to install on a Linux-machine with CPU (no GPU), the following commands installs the required packages
```
# PyTorch
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
# PyTorch Geometric
pip install torch_geometric
# PyTorch Cluster
pip install torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cpu.html
pip install torchinfo lightning
```

### Initial imports
First, some initial imports from used libraries (might take some extra time the first time when caching).

In [1]:
from argparse import Namespace
import matplotlib.pyplot as plt
import matplotlib

import torch
from torchinfo import summary
from lightning.pytorch import seed_everything

from trajectory_prediction.metrics import MinFDE, MinADE, MissRate
from trajectory_prediction.datamodules.dataloader import DroneDataModule
from trajectory_prediction.plot_tools import plot_scenario, plot_heatmap, plot_scenario_map, get_agent_positions
from trajectory_prediction.utils import load_pretrained_model

/home/jessy/Polytech_Grenoble/INFO5/S9 - Linköping/Autonomous Vehicles and Systems - TSFS12/hands-in/TSFS12-Autonomous_Vehicles/handin5/tsfs12-master-Handin_Exercises-HI5-Learning/Handin_Exercises/HI5-Learning/python/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Run the command below if you want your plots in external windows, e.g., with the possibility to zoom. Alternatively, skip the line and generate all plots as static images in the notebook. It is recommended during exploration, since then you can interact and zoom with the plots.

In [2]:
# choose another matplotlib backend for plotting if you want
%matplotlib tk

## 1. Data exploration

When working with data-driven modelling techniques, data-exploration and an understanding of the data is essential. This is often a process that takes considerable time and effort and therefore this is also an important part also in this exercise. 

This section of the exercise will therefore be devoted to loading the data and explore it to get some basic understanding of the data. It is not necessary that you understand all the details of the data, however the more you understand of the data the easier it is to understand what information the predictive models have and understanding the prediction output. 

### 1.1 Load the data

The first step is to load the evaluation data and prepare a data loader to make it easy to iterate through the data. 

Before you begin, set the path to the where the datasets are located. If you do thin in the university Linux-labs, they are located at `/courses/tsfs12/trajectory_prediction/datasets`. If you do this exercise on your own computer, you need to download the datasets from Lisam, there is a zip-file in the Course Documents. The unzipped data is approximately 1.3 GB.

In [3]:
dataset_path = "datasets"  # if you have the datasets unzipped in the current folder
# dataset_path = "/courses/tsfs12/trajectory_prediction/datasets"  # if you are in the linux labs

Prepare the data loaders.

In [4]:
seed = 42 # A random seed is set so that all environments gives the same results. Do not change the value 42 (at least not in a first run).
seed_everything(42)

# Define the step size 0.2 seconds per sample
STEP_SIZE = 0.2

# Create the datamodule and load the data
config = {
    "root": dataset_path,
    "batch_size": 32,
    "transform": "CoordinateTransform",
    "name": "sinD",
}
args = Namespace(
    small_ds=False,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
    seed=seed,
    eval_only=True,
)
dm = DroneDataModule(config, args)
dm.setup()

# Get the test dataloader
loader = dm.test_dataloader()
print(f"Data is loaded and there are {len(loader)} batches in the test dataset")

Seed set to 42


Data is loaded and there are 279 batches in the test dataset


### 1.2 What is in a batch of data?

As seen above, the test data consists of a set of _batches_ and each batch consists of a set of _scenarios_. In this case each batch consists of 32 scenarios. A single batch is represented using a `HeteroDataBatch` object from the [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/) library, built for graph-learning applications but is also useful in learning problems where the data points (samples) in our datasets have varying shapes (dimensions). This is true in many motion prediction problems where we typically encounter a variable number of agents present in each scenario. It is not important for the exercise to be familiar with PyTorch Geometric.

A batch object works similarly like a Python dictionary, or a struct in Matlab, meaning we can access specific properties by indexing the object using the corresponding `key`. In this exercise, a typical batch object may look like:

```
HeteroDataBatch(
  rec_id=[32],
  agent={
    num_nodes=885,
    ta_index=[32],
    ids=[32],
    type=[885],
    inp_pos=[885, 15, 2],
    inp_vel=[885, 15, 2],
    inp_acc=[885, 15, 2],
    inp_yaw=[885, 15, 1],
    trg_pos=[885, 25, 2],
    trg_vel=[885, 25, 2],
    trg_acc=[885, 25, 2],
    trg_yaw=[885, 25, 1],
    input_mask=[885, 15],
    valid_mask=[885, 25],
    sa_mask=[885, 25],
    ma_mask=[885, 25],
    batch=[885],
    ptr=[33],
  },
  map_point={
    num_nodes=13088,
    type=[13088],
    position=[13088, 2],
    batch=[13088],
    ptr=[33],
  },
  (map_point, to, map_point)={
    edge_index=[2, 27328],
    type=[27328, 1],
  }
)
```

First, let us get the first batch object in the test set to explain the contents of the data.

In [5]:
generator = iter(loader)
data = next(generator)
data

HeteroDataBatch(
  rec_id=[32],
  agent={
    num_nodes=885,
    ta_index=[32],
    ids=[32],
    type=[885],
    inp_pos=[885, 15, 2],
    inp_vel=[885, 15, 2],
    inp_acc=[885, 15, 2],
    inp_yaw=[885, 15, 1],
    trg_pos=[885, 25, 2],
    trg_vel=[885, 25, 2],
    trg_acc=[885, 25, 2],
    trg_yaw=[885, 25, 1],
    input_mask=[885, 15],
    valid_mask=[885, 25],
    sa_mask=[885, 25],
    ma_mask=[885, 25],
    batch=[885],
    ptr=[33],
  },
  map_point={
    num_nodes=13088,
    type=[13088],
    position=[13088, 2],
    batch=[13088],
    ptr=[33],
  },
  (map_point, to, map_point)={
    edge_index=[2, 27328],
    type=[27328, 1],
  }
)

Before describing the data structure, let's plot one of the scenarios in the first batch. Where you can see a set of vehicles in a specific situation in an intersection. Remember these commands, you will need them later. 

In [6]:
fig, ax = plt.subplots(num=1, clear=True)
plot_scenario(ax, data, highlight_idx=None, scenario_idx=10)
fig.savefig("scenario_plot_example.pdf", dpi=300)

To save a figure to file for inclusion in your report, do
```python
fig.savefig("filename.pdf")
```
to save in PDF format or if you prefer PNG format you instead run
```python
fig.savefig("filename.pdf")
```
The variable `fig` is the one created with the `plt.subplots` command for the figure yoou wanted to save.


Going back to the batch-data, a single object hold several pieces of information, of notable interest is the contents of the property "`agent`". If you index `data` with the key `agent` as
```python
data['agent']
```
it will return a nested dictionary that can be similarly accessed as its parent by using a key, e.g., using `inp_pos`, `inp_vel`, and so on.

This object holds information on the input (preceeded by `inp_`) and target (preceeded by `trg_`) features of the agents in each scenario. Note that each feature is three-dimensional, e.g., `inp_pos`  have shape `(N, T, 2)`.
Here, `N` refers to the total number of agents in the batch, which is the sum of all agents across all scenes in the batch, `T=15` is the length of the input sequence, and final dimension represents positions in 2D (x, y coordinates). 

The sampling time is $0.2$ s and since the input and target are 15 and 25 samples respectively, this corresponds to $3$ and $5$ seconds respectively.


In [8]:
print(f"Number of vehicles in total for all scenarios in the batch: {data['agent']['num_nodes']}")

Number of vehicles in total for all scenarios in the batch: 885


To get the input to the prediction for the first vehicle in the first batch (index 0), run the command below extracting the `inp_pos` property. You will get 15 pairs of (x, y) coordinates, and since the sampling rate is 0.2s this corresponds to 3 seconds of historical data. To get the corresponding for velocity and acceleration extract the `inp_vel` or `inp_acc` properties. There are also `trg_pos`, `trg_vel`, `trg_acc` for the prediction targets, i.e., 5 seconds of pos/vel/acc following the input.

In [7]:
data['agent']['inp_pos'][0]

tensor([[-0.0073,  0.0060],
        [-0.0007, -0.0033],
        [ 0.0209, -0.0038],
        [ 0.0451, -0.0062],
        [ 0.0730, -0.0060],
        [ 0.0834, -0.0035],
        [ 0.0769,  0.0017],
        [ 0.0554,  0.0039],
        [ 0.0313,  0.0043],
        [ 0.0260,  0.0047],
        [ 0.0237,  0.0061],
        [ 0.0042,  0.0069],
        [-0.0079,  0.0053],
        [-0.0096,  0.0027],
        [ 0.0000,  0.0000]])

A very important key-value pair in the `agent` dictionary is `batch`. The contents of 
```python
data['agent']['batch']
```
indicate which agents belong to the same scenario. In our current configuration, we use a batch size of 32, meaning that each batch includes 32 unique scenarios from our test set. This means that `data['agent']['batch']` contains integers from 0 to 31 which can be used to map the agent features to the correct scenario. 
To see this in practice, create a boolean mask to get all the indices that belong to scenario 0 using the command below. The number of vehicles in a scenario are the number of vehicles present during the 3 + 5 second time-window.


In [9]:
mask = (data['agent']['batch'] == 0)
print(f"The number of vehicles in scenario 0: {mask.sum()}")

The number of vehicles in scenario 0: 20


We can use this boolean mask to slice our data and extract features that belong to scenario 0 as:

In [10]:
inp_pos_first_scenario = data['agent']['inp_pos'][mask]
trg_pos_first_scenario = data['agent']['trg_pos'][mask]
print(inp_pos_first_scenario.shape, trg_pos_first_scenario.shape)

torch.Size([20, 15, 2]) torch.Size([20, 25, 2])


### 1.3 Map features
We also note that there is map information in the form of lane graphs for each scenario representing the road network. This is here primarily used for visualization purposes, but two of the used models encode also the map information and utilizes that when making trajectory predictions.


The following code illustrate how you can plot scenarios, in a 2 x 2 grid, to show data in the test set.

In [11]:
# %% Plot 4 scenarios in the first batch
scenarios_to_plot = [0, 1, 2, 4]  # 32 scenarios in a batch
fig, ax = plt.subplots(2, 2, num=10, clear=True, layout="constrained")
for k, scenario_idx in enumerate(scenarios_to_plot):
    mask = data["agent"]["batch"] == scenario_idx
    inp_pos = data["agent"]["inp_pos"][mask]

    r, c = k // 2, k % 2
    plot_scenario(ax[r, c], data, highlight_idx=None, scenario_idx=scenario_idx)
    ax[r, c].legend().remove()
_ = ax[1, 1].legend()

#fig.savefig("multiple_scenarios_plot_example.pdf", dpi=300)

### 1.4 Questions on the data exploration

To verify you understand the data structure, know how to access specific agent features, a few control questions:

- How many batches and how many scenarios are there in total in the test dataset?
- How many agents are present in the first batch (already available in the variable `data`)?
- How many agents are present in scenario 1 and 2 respectively in the first batch?
- How many different agent types can you find in the batch (look at the `type` key)?
- Extract the target (ground truth) positions for the first agent in the data batch (similar as was done for the input positions earlier).
- Determine how many vehicles are involved in scenario 20 in the first batch.

In [12]:
#question 1
num_batches = len(loader)
num_scenarios = num_batches * 32  #32 scenarios per batch
print(f"Total batches: {num_batches}")
print(f"Total scenarios: {num_scenarios}")

#question 2
num_agents_batch = data['agent']['num_nodes']
print(f"Number of agents in first batch: {num_agents_batch}")

# question 3
mask_scenario_1 = (data['agent']['batch'] == 1)
mask_scenario_2 = (data['agent']['batch'] == 2)
print(f"Number of agents in scenario 1: {mask_scenario_1.sum()}")
print(f"Number of agents in scenario 2: {mask_scenario_2.sum()}")

# question 4
agent_types = data['agent']['type'].unique()
print(f"Number of different agent types: {len(agent_types)}")
print(f"Agent types: {agent_types}")

#question 5
first_agent_target = data['agent']['trg_pos'][0]
print(f"Target positions shape for first agent: {first_agent_target.shape}")
print(f"First few target positions:\n{first_agent_target[:5]}")

#Question 6
mask_scenario_20 = (data['agent']['batch'] == 20)
num_agents_scenario_20 = mask_scenario_20.sum()
print(f"Number of vehicles in scenario 20: {num_agents_scenario_20}")

Total batches: 279
Total scenarios: 8928
Number of agents in first batch: 885
Number of agents in scenario 1: 20
Number of agents in scenario 2: 20
Number of different agent types: 4
Agent types: tensor([0, 1, 3, 5])
Target positions shape for first agent: torch.Size([25, 2])
First few target positions:
tensor([[ 0.0345, -0.0030],
        [ 0.1178, -0.0098],
        [ 0.2807, -0.0127],
        [ 0.5522, -0.0106],
        [ 0.9288, -0.0019]])
Number of vehicles in scenario 20: 34


**Answers:**
1. **Total batches and scenarios**: 279 batches × 32 scenarios/batch = 8928 total scenarios
2. **Agents in first batch**: 885 agents
3. **Agents in scenarios 1 and 2**: 20 agents in both scenarios
4. **Agent types**: 0, 1, 3, 5
5. **Target positions**: 25 time steps × 2 coordinates (x, y)
6. **Vehicles in scenario 20**: 34 vehicles.

To illustrate data available for prediction for one single vehicle, let's plot the history and target trajectory for agents with idx 12 and 20 in scenario 10 in the first batch. Here the `get_agent_positions` function is used to extract positional data.

In [13]:
scenario_idx = 13

scenario_mask = data["agent"]["batch"] == scenario_idx
inp_pos_12, trg_pos_12 = get_agent_positions(data, scenario_idx, 12)
inp_pos_20, trg_pos_20 = get_agent_positions(data, scenario_idx, 20)

fig, ax = plt.subplots(num=1, clear=True, layout="constrained")
plot_scenario_map(ax, data, scenario_idx)
ax.plot(inp_pos_12[:, 0], inp_pos_12[:, 1], label="Observed", color="blue")
ax.plot(trg_pos_12[:, 0], trg_pos_12[:, 1], label="Target", color="red")
ax.plot(inp_pos_20[:, 0], inp_pos_20[:, 1], color="blue")
ax.plot(trg_pos_20[:, 0], trg_pos_20[:, 1], color="red")
ax.set_title(f"Scenario {scenario_idx}, Agents 12 and 20")
_ = ax.legend(frameon=False)
#fig.savefig("scenario_10_agents_12_20_plot.pdf", dpi=300)

/tmp/ipykernel_4727/3271948123.py:7: UserWarning: Ignoring specified arguments in this call because figure with num: 1 already exists
  fig, ax = plt.subplots(num=1, clear=True, layout="constrained")


The `plot_scenario` function has an argument `highlight_idx` that is useful to highlight particular agents/vehicles in a given scenario. For example, the code below plots all vehicles in scenario 10, and highlight agents with idx (20, 12, 15).

Experiment with different scenarios and `highlight_idx` to find interesting cases. (It might be easier if you plot in an external window and use the zoom functionality).

In [14]:
agent_idx = 20
fig, ax = plt.subplots(num=2, clear=True)
plot_scenario(ax, data, highlight_idx=[agent_idx, 12, 15], scenario_idx=scenario_idx)
plt.show()
#fig.savefig("scenario_13_highlighted_agents_plot.pdf", dpi=300)

## 2. Inspect the models and make some predictions

Now, let's look at the provided models. There are 4 models provided, 3 neural network models (pre-trained) and 1 simple constant velocity model. The three neural network models are
* Neural Network with no utilization of the map
* Neural Network with map encoder
* Neural Network with a more extensive map encoder, Graph Attention Network (GAT)

### 2.1 Load the models
First, load the pre-trained model and print out some general information.

In [15]:
model_nomap = load_pretrained_model()
_ = model_nomap.eval()  # Set the module in evaluation mode, no training in the assignment
summary(model_nomap.model)

Layer (type:depth-idx)                                  Param #
TrajNet                                                 --
├─Linear: 1-1                                           1,024
├─GRU: 1-2                                              99,072
├─InteractionNet: 1-3                                   --
│    └─MultiheadAttention: 2-1                          49,536
│    │    └─NonDynamicallyQuantizableLinear: 3-1        16,512
├─GRUCell: 1-4                                          50,688
├─Linear: 1-5                                           258
Total params: 217,090
Trainable params: 217,090
Non-trainable params: 0

In [16]:
model_map = load_pretrained_model(extra="map_mpnn")  # Map-based model
_ = model_map.eval()
summary(model_map.model)

Layer (type:depth-idx)                                  Param #
TrajNet                                                 --
├─Linear: 1-1                                           1,024
├─GRU: 1-2                                              99,072
├─InteractionNet: 1-3                                   --
│    └─MultiheadAttention: 2-1                          49,536
│    │    └─NonDynamicallyQuantizableLinear: 3-1        16,512
├─MPNNMapEncoder: 1-4                                   --
│    └─Sequential: 2-2                                  --
│    │    └─GraphConv: 3-2                              640
│    │    └─GELU: 3-3                                   --
│    │    └─GraphConv: 3-4                              16,448
│    │    └─GELU: 3-5                                   --
│    │    └─GraphConv: 3-6                              16,512
│    └─Sequential: 2-3                                  --
│    │    └─Linear: 3-7                                 896
│    │    └─GELU: 3-8     

In [17]:
model_gatmap = load_pretrained_model(extra="map_gat")  # Map-based model
_ = model_gatmap.eval()
summary(model_gatmap.model)

Layer (type:depth-idx)                                  Param #
TrajNet                                                 --
├─Linear: 1-1                                           1,024
├─GRU: 1-2                                              99,072
├─InteractionNet: 1-3                                   --
│    └─MultiheadAttention: 2-1                          49,536
│    │    └─NonDynamicallyQuantizableLinear: 3-1        16,512
├─GATMapEncoder: 1-4                                    --
│    └─Sequential: 2-2                                  --
│    │    └─GATv2Conv: 3-2                              1,792
│    │    └─GELU: 3-3                                   --
│    │    └─GATv2Conv: 3-4                              34,048
│    │    └─GELU: 3-5                                   --
│    │    └─GATv2Conv: 3-6                              34,048
├─Sequential: 1-5                                       --
│    └─Linear: 2-3                                      32,896
│    └─GELU: 2-4     

And also define our constant velocity prediction model.

In [22]:
def constant_velocity(data, N, dt):
    x0 = data["agent"]["inp_pos"][:, -1:]  # Last observed position
    v0 = data["agent"]["inp_vel"][:, -1:]  # Last observed velocity

    t = torch.arange(1, N + 1, device=v0.device) * dt 
    t = t.reshape(1, -1, 1)  # Ensure broadcasting works correctly

    return x0 + v0 * t  # Euler forward

### 2.2 A first prediction
Now, let us make a prediction for all vehicles in a given batch of data using the simplest of the neural network models and plot the result.

In [ ]:
with torch.no_grad():  # Turn of gradients computation
    _, pred, attn, _ = model_gatmap(data) # other models

fig, ax = plt.subplots(num=15, clear=True, layout="constrained")
plot_scenario(ax, data, pred, scenario_idx=scenario_idx, highlight_idx=[agent_idx, 12, 15])


In [23]:
with torch.no_grad():  # Turn of gradients computation
    cv_pred = constant_velocity(data, 25, STEP_SIZE) # for constnant velocity

fig, ax = plt.subplots(num=15, clear=True, layout="constrained")
plot_scenario(ax, data, cv_pred, scenario_idx=scenario_idx, highlight_idx=[agent_idx, 12, 15])

/tmp/ipykernel_4727/3245601329.py:4: UserWarning: Ignoring specified arguments in this call because figure with num: 15 already exists
  fig, ax = plt.subplots(num=15, clear=True, layout="constrained")


Here we've made use of the `highlight_idx` argument, that highligts given agents in the plot for easier illustration.

To get the predicted trajectory of an agent, 

In [24]:
scenario_idx = 10
agent_idx = 20
scenario_mask = data["agent"]["batch"] == scenario_idx
cv = True # set to True to use constant velocity model
if cv:
    pred = cv_pred
agent_prediction = pred[scenario_mask][agent_idx]  # Get the prediction for the agent
print(f"Matrix with predicted (x,y)-coordinates has dimension {agent_prediction.shape}")

Matrix with predicted (x,y)-coordinates has dimension torch.Size([25, 2])


Below is code for plotting a single agent history, target, and prediction.

In [25]:
inp_pos, trg_pos = get_agent_positions(data, scenario_idx, agent_idx)

inp_pos_12, trg_pos_12 = get_agent_positions(data, scenario_idx, 12)  
agent_prediction_12 = pred[scenario_mask][12]

inp_pos_15, trg_pos_15 = get_agent_positions(data, scenario_idx, 15)  
agent_prediction_15 = pred[scenario_mask][15]

fig, ax = plt.subplots(num=15, clear=True, layout="constrained")
plot_scenario_map(ax, data, scenario_idx)
ax.plot(inp_pos[:, 0], inp_pos[:, 1], label="Observed", color="blue")
ax.plot(trg_pos[:, 0], trg_pos[:, 1], label="Target", color="red")
ax.plot(agent_prediction[:, 0], agent_prediction[:, 1], label="Predicted", color="green")

ax.plot(inp_pos_12[:, 0], inp_pos_12[:, 1], color="blue")
ax.plot(trg_pos_12[:, 0], trg_pos_12[:, 1], color="red")
ax.plot(agent_prediction_12[:, 0], agent_prediction_12[:, 1], color="green")

ax.plot(inp_pos_15[:, 0], inp_pos_15[:, 1], color="blue")
ax.plot(trg_pos_15[:, 0], trg_pos_15[:, 1], color="red")
ax.plot(agent_prediction_15[:, 0], agent_prediction_15[:, 1], color="green")

ax.set_title(f"Scenario {scenario_idx}, Agents 12, 15 and {agent_idx}")
_ = ax.legend(frameon=False)

fig.savefig("tagrandmere.pdf", dpi=300)

/tmp/ipykernel_4727/4247978612.py:9: UserWarning: Ignoring specified arguments in this call because figure with num: 15 already exists
  fig, ax = plt.subplots(num=15, clear=True, layout="constrained")


### 2.3 Timing

Run the cells below to time the different methods. Write down the values you get to be used in the report. Note that runtime will vary from machine to machine. Especially if you have GPU support.


In [27]:
%%timeit -n 10 -r 5
with torch.no_grad():  # Turn off gradients computation
    _, pred, _, _ = model_nomap(data)

70.5 ms ± 4.95 ms per loop (mean ± std. dev. of 5 runs, 10 loops each)


In [24]:
%%timeit  -n 10 -r 5
with torch.no_grad():  # Turn off gradients computation
    _, pred, _, _ = model_map(data)

217 ms ± 8.53 ms per loop (mean ± std. dev. of 5 runs, 10 loops each)


In [25]:
%%timeit  -n 10 -r 5
with torch.no_grad():  # Turn off gradients computation
    _, pred, _, _ = model_gatmap(data)

757 ms ± 54.4 ms per loop (mean ± std. dev. of 5 runs, 10 loops each)


In [26]:
%%timeit  -n 1000 -r 5
cv_pred = constant_velocity(data, 25, STEP_SIZE)

204 μs ± 13.1 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)


### 2.4 Questions for the models and prediction section

- Explore the first data batch and find interesting scenarios and agents, use provided code (use the `highlight_idx` parameter). 
    - Start with agents with index 12, 15, and 20 in scenario 10. 
    - Compare the 4 different prediction models and reflect upon the results. 
    - In particular, think about predictions in cases there are several candidate future options, e.g., left-right-straight
    - Think about situations where the constant-velocity model might be expected to work well and where it doesn't.
- Include interesting plots in the report that you found during your exploration.

Inspect the architecture
- How many parameters are in each model, relate to the inference/prediction time.
- Discuss prediction time with respect to a real-time application of the model in a real vehicle. Which models are real-time feasible?
- Optional if you have sufficient python/pytorch experience:
    - How many components does the model consist of and what is the purpose of each component?

**Key observations:**

1. **Model Comparison on Different Scenarios:**
   - **Constant Velocity (CV)**: Works well for straight-line motion at constant speed, but fails at intersections where vehicles turn, stop, or change lanes
   - **No-map model**: Better handles turns and interactions but may struggle with understanding lane constraints
   - **Map-based models**: Better respect road geometry and lane structure
   - **GAT model**: Best at capturing complex interactions between agents

2. **Multi-modal situations** (left-right-straight):
   - Neural networks tend to predict an "average" trajectory when multiple futures are possible
   - This can result in predictions that go through the middle of an intersection when the vehicle could go left, right, or straight

3. **Model Parameters vs. Inference Time:**
   - **CV model**: ~0 parameters, fastest (< 1 ms) : 204 μs ± _13.1 μs_ per loop
   - **No-map model**: 217,090 parameters : **70.5 ms** ± _4.95 ms_ per loop
   - **Map model**: 301,123 parameters : **217 ms** ± _8.53 ms_ per loop
   - **GAT map**: 336,386 parameters (most complex, likely slowest) : **757 ms** ± _54.4 ms_ per loop
   
4. **Real-time feasibility:**
   - **Constant velocity** : highly feasable
   -  **No-map Model** : likely feasable
   - **Map Model** : Marginal/not feasable
   - **GAT Model** : Not feasable
   - Neural network models depend on your hardware (CPU vs GPU). These are the conclusion I had on my laptop.

## 3. Explore the Attention Mechanism

It is clearly the case that the interaction between agents directly impact their behavior; if a vehicle in front is braking you should probably do the same to avoid collision. Therefore, including interaction mechanisms in the prediction models gives significant performance improvements.
The attention mechanism in the Transformers architecture is one suitable way to learn the interactions in-between agents in a scenario.
For more details on the Transformers and attention in general, see

Bishop, C. M., & Bishop, H. (2023). "_Deep learning: Foundations and concepts_". Springer Nature (https://www.bishopbook.com)

The attention mechanism is a key component in many deep learning models, and it allows the model to focus on specific parts of the input data when making predictions. The attention mechanism is based on the concept of attention weights, which determine the importance of each element in the input data.
We will explore in this section how analyzing the attention weights, we can gain insights into which agents are important for the prediction and how they influence each other.

The attention mechanism for agent-to-agent interactions is best described using the Graph Attention Network (GAT) architecture. In this architecture, each agent is represented as a node in a graph, and the attention weights are learned based on the interactions between the nodes. In the image below, you can see an example of the attention mechanism in a GAT model.
Here $h_i$ represents the hidden (latent) state of agent $i$, and $h_j$ represents the hidden state of agent $j$. The attention weights $a_{ij}$ represent the importance of agent $j$ for agent $i$.
By combining the attention from all interacting agents, we then update the hidden state of agent $i$ in order to get $h_i'$.

In the example image below, we have $6$ different nodes.
The attention matrix is a $6 \times 6$ matrix, where each row corresponds to the attention weights of one agent (query) towards all other agents (keys), including itself.

<div style="text-align: center;">
    <img src="media/attn_mech.png" alt="Graph Attention" style="width: 40%;">
</div>

Now, make a prediction with the simplest neural network model

In [26]:
with torch.no_grad():  # Turn of gradients computation
    _, pred, attn, _ = model_nomap(data)

To illustrate the intensity of the attention between agents, the code below plots the scenario and a heatmap of the attention matrix of all agents.

In [27]:
scenario_idx = 2
highlight_idx = [13, 5]  # Sample agents to highlight

fig, ax = plt.subplots(num=30, clear=True, layout="constrained")
plot_scenario(ax, data, pred, scenario_idx=scenario_idx, highlight_idx=highlight_idx)
plt.savefig("3_scenario_plot_with_predictions.pdf", dpi=300)

fig, ax = plt.subplots(num=20, clear=True, layout="constrained")
plot_heatmap(ax, data, attn, scenario_idx=scenario_idx)
plt.savefig("3_attention_matrix_heatmap.pdf", dpi=300)

Study the heatmap to understand which parts of the input data the model focuses on when making predictions.
The diagonal elements of the heatmap represent the self-attention of each agent, while the off-diagonal elements represent the attention between agents.
An agent (*query*) attends to other agents (*keys*) based on their perceived importance for the prediction.
 
The indices correspond to the agents in the scenario, which are the same as the indices in the plot_scenario function.


### Answers for 3.

- **Dominant Self-Attention** : The most striking pattern is the strong dark red diagonal. This indicates that every agent primarily focuses on its own past history when generating a prediction (Self-Attention is much stronger than cross-agent attention). For agents without immediate conflicts, the simplest and most effective prediction is often based on an extrapolation of their own recent motion (speed and heading).

- **Limited and Localized Cross-Attention** : Off-diagonal attention, which represents social interaction, is generally weak (light yellow to orange).

- **Leading/Following Interactions** : Where cross-attention does occur, it appears highly localized, typically for agents that are numerically close in index. For example:

    - Agent 3 attends strongly to Agent 4.

    -  Agent 18 attends to Agent 19.

    This suggests the model is learning local interactions, such as an agent following another closely in the same lane, where the trailing vehicle must attend to the leading vehicle's deceleration.

- **Agents with Sparse Attention**: Many agents, such as Agent 6 or Agent 14, show very little off-diagonal attention, suggesting they are either isolated, moving on a clear path, or simply not near any other influential agent in this specific scenario.

### 3.1 Questions and assignments

Your task is to find patterns in the heat map and relate them to the scenario under investigation.
Find examples of agents that are highly attended to and agents that are not attended to.
Use the `plot_scenario` function to visualize the scenario and identify the agents in the heatmap.
- Does the learned attention make sense given the scenario?
    - Search for different interesting scenarios, start with scenarios 2 and 23. Plot the heatmap and identify interesting agents.
- Does it correspond to your intuition of which agents are important for the prediction?
- Discuss your observations


### Answers on 3.1

- **Does the learned attention make sense given the scenario?**

    Yes, the learned attention generally aligns with intuition. In traffic, an agent's future movement is mostly determined by its own inertia and intent (high self-attention). Interaction only becomes critical when vehicles are close and their paths conflict (localized cross-attention). The heatmap suggests that the model effectively learns this hierarchy: prioritize your own movement, then locally adjust based on the most immediate neighbor.

- **Does it correspond to your intuition of which agents are important for the prediction?**

    It corresponds well for local interactions. A high attention from Agent i to Agent j implies j is a perceived threat or constraint for i. Agents that are far apart, or those traveling on completely non-conflicting paths, are correctly ignored (low attention).

- **Discussion**

    The strong diagonal highlights a limitation of the model: it heavily relies on trajectory extrapolation and may not be sufficiently leveraging social interactions unless the relationship is very clear and close.

    The general sparsity of the attention matrix (most off-diagonal entries are near zero) suggests that the model is efficiently filtering out non-relevant agents, which is a key goal of the attention mechanism.

## 4. Explore and evaluate the different models

In this final part, you should investigate the performance of the different methods. 
The cells below will evaluate the performance for each batch and print out the mean performance across the test data.
This takes a while, could be a few minutes. Therefore, if you set `full_eval` to `False` below, the loop will break after the first batch, which could be helpful if you only want to quickly test the performance. Set `full_eval` to `True` when you collect results for your report. 

Three different performance measures will be evaluated
* average displacement error (ADE),
* final displacement error (FDE)
* miss rate
The ADE measures the average displacement error between the predicted and true positions of the agents, while the FDE measures the displacement error of the agent's final position.

For a full description of the evaluation metrics, see Section IV in the paper "_Toward Unified Practices in Trajectory Prediction Research on Drone Datasets_" 
(https://arxiv.org/abs/2405.00604).

In [ ]:
pred_hrz = data["agent"]["trg_pos"].shape[1]  # Prediction horizon (25 steps)
cv_pred = constant_velocity(data, pred_hrz, STEP_SIZE)

fig, ax = plt.subplots(num=40, clear=True, layout="constrained")
plot_scenario(ax, data, cv_pred)
#plt.savefig("tagrandmere.pdf", dpi=300)

In [31]:
# %% Compare performance between the neural net and the CV model
# Initalize torchmetrics objects for FDE, ADE, and Miss Rate for both ML and CV models
fde_nomap, ade_nomap, miss_rate_nomap = MinFDE(), MinADE(), MissRate()
fde_map, ade_map, miss_rate_map = MinFDE(), MinADE(), MissRate()
fde_gatmap, ade_gatmap, miss_rate_gatmap = MinFDE(), MinADE(), MissRate()
fde_cv, ade_cv, miss_rate_cv = MinFDE(), MinADE(), MissRate()

full_eval = True  # set to 'False' if we only want to study performance over a single batch

pred_hrz = data["agent"]["trg_pos"].shape[1]  # Prediction horizon (25 steps)

# Compute ADE and FDE for the neural model predictions
for data in loader:  # Loop over all mini-batches in the test set
    target_pos = data["agent"]["trg_pos"]
    mask = data["agent"]["ma_mask"]

    # Constant velocity predictions
    cv_pred = constant_velocity(data, pred_hrz, STEP_SIZE)

    # Neural network predictions
    with torch.no_grad():  # Turn of gradients computation
        _, eval_pred, _, _ = model_nomap(data)
        _, eval_pred_map, _, _ = model_map(data)
        _, eval_pred_gatmap, _, _ = model_gatmap(data)

    # Update prediction errors for all 4 models
    ade_nomap.update(eval_pred, target_pos, mask=mask)
    fde_nomap.update(eval_pred, target_pos, mask=mask)
    miss_rate_nomap.update(eval_pred, target_pos, mask=mask)

    ade_map.update(eval_pred_map, target_pos, mask=mask)
    fde_map.update(eval_pred_map, target_pos, mask=mask)
    miss_rate_map.update(eval_pred_map, target_pos, mask=mask)

    ade_gatmap.update(eval_pred_gatmap, target_pos, mask=mask)
    fde_gatmap.update(eval_pred_gatmap, target_pos, mask=mask)
    miss_rate_gatmap.update(eval_pred_gatmap, target_pos, mask=mask)

    ade_cv.update(cv_pred, target_pos, mask=mask)
    fde_cv.update(cv_pred, target_pos, mask=mask)
    miss_rate_cv.update(cv_pred, target_pos, mask=mask)

    if not full_eval:  # if we only want to study performance over a single batch
        break

ml_ade, ml_fde, ml_mr = ade_nomap.compute(), fde_nomap.compute(), miss_rate_nomap.compute()
ml_map_ade, ml_map_fde, ml_map_mr = ade_map.compute(), fde_map.compute(), miss_rate_map.compute()
ml_gatmap_ade, ml_gatmap_fde, ml_gatmap_mr = ade_gatmap.compute(), fde_gatmap.compute(), miss_rate_gatmap.compute()
cv_ade, cv_fde, cv_mr = ade_cv.compute(), fde_cv.compute(), miss_rate_cv.compute()

print(f"Constant velocity:: ADE: {cv_ade:.2f} m | FDE: {cv_fde:.2f} m | Miss Rate: {cv_mr:.2f}")
print(f"ML model, no map :: ADE: {ml_ade:.2f} m | FDE: {ml_fde:.2f} m | Miss Rate: {ml_mr:.2f}")
print(f"ML model, map    :: ADE: {ml_map_ade:.2f} m | FDE: {ml_map_fde:.2f} m | Miss Rate: {ml_map_mr:.2f}")
print(f"ML model, GAT map:: ADE: {ml_gatmap_ade:.2f} m | FDE: {ml_gatmap_fde:.2f} m | Miss Rate: {ml_gatmap_mr:.2f}")


Constant velocity:: ADE: 1.89 m | FDE: 4.99 m | Miss Rate: 0.51
ML model, no map :: ADE: 1.20 m | FDE: 3.30 m | Miss Rate: 0.46
ML model, map    :: ADE: 1.09 m | FDE: 2.96 m | Miss Rate: 0.44
ML model, GAT map:: ADE: 1.05 m | FDE: 2.86 m | Miss Rate: 0.43


### 4.1 Questions for the model evaluations

In your report, reflect on the various metrics used in the assignment and evaluate the overall performance of the models. Consider addressing the following questions:

- What do the different displacement metrics measure? Are both of them necessary?
- How do the various methods compare? Is the performance difference significant?
- How does the improved performance of ML-based approaches affect overall runtime? Is their use justified?
- Does incorporating map-based information improve performance? Why or why not?
